# SImulations

In this notebook we simulate experiments to prove the efficacy of the Softmin Energy minimization method over other established global optimization methods.
The Softmin energy is defined as 
$$
    J_\beta(\mathbf{x}) = \sum_{i=1}^n \frac{e^{-\beta f (x_i)}f(x_i)}{\sum_{j=1}^n e^{-\beta f(x_j)}}
$$
with $\beta>0$, $\mathbf{x} = (x_1,\dots,x_n)$.

## Imports and utility functions

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle as pkl

from tqdm import tqdm
from simulations import *
from potentials import *
from viz import plot_entry_time_comparison, plot_value_comparison, plot_prefactor_dynamics
import multiprocessing as mp

plt.rcParams['font.size'] = 15
#uncomment to double the fontsize in plots

N_CPU = mp.cpu_count()


## Exit time from the maximizing regime

### Plot

In [ ]:
from scipy.optimize import brentq

noise = 0.05
beta = 2
n_particles = 100
tilt = 0.05  # matches TILT in run_theory_verification.py

def u(r, beta, n=n_particles):
    """Reduced potential along the frozen-background channel (Theorem 3.1, revised theory)."""
    return n * r * np.exp(-beta * r) / (n - 1 + np.exp(-beta * r))

def solve_r_c(beta, n=n_particles):
    """r_c = argmax u(r) = unique positive root of q(r) = (n-1)(1-beta*r) + exp(-beta*r)."""
    q = lambda r: (n - 1) * (1 - beta * r) + np.exp(-beta * r)
    return brentq(q, 1e-9, 50 / beta)

# beta and n are fixed for this whole sweep, so r_c is a single constant: the bound u(min(h, r_c))
# saturates once the physical barrier h exceeds r_c (Remark: "barrier saturates rather than
# growing with h").
r_c = solve_r_c(beta)

mean_exit_times, theory_bounds, x = [], [], []
q1_exit_times, q3_exit_times, std_exit_times = [], [], []

for delta in np.arange(0.25, 4.25, 0.125):
    # asymmetric_double_well is used both here (to compute h) and by run_theory_verification.py
    # (to generate exit_times below), so this overlay is a genuine apples-to-apples check against
    # Theorem 3.1's setup -- run `python run_theory_verification.py` on the cluster first if
    # results/softmin_static_double_well_init0_*.pkl don't exist yet.
    kwargs = asymmetric_double_well(delta, tilt)
    h = float(kwargs['f_vec']([kwargs['saddle']]) - kwargs['f_vec']([kwargs['z_start']]))
    if delta.is_integer(): delta = int(delta)
    string = f"softmin_static_double_well_init0_{delta}_{noise}".replace('.','p')+".pkl"
    with open(f"results/{string}", "rb") as f:
        exit_times = pkl.load(f)

    x.append(h)
    mean_exit_times.append(np.mean(exit_times))
    theory_bounds.append(np.exp(u(min(h, r_c), beta) / noise))

    std_exit_times.append(np.std(exit_times))
    q1_exit_times.append(np.percentile(exit_times, 25))
    q3_exit_times.append(np.percentile(exit_times, 75))

def get_asymmetric_errorbars(mean, q1, q3):
    lower = np.array(mean) - np.array(q1)
    upper = np.array(q3) - np.array(mean)
    return [np.maximum(lower, 0), np.maximum(upper, 0)]


fig, ax = plt.subplots(figsize=(10, 6))
err_exit_times = get_asymmetric_errorbars(mean_exit_times, q1_exit_times, q3_exit_times)
ax.plot(x, mean_exit_times, label='Empirical Times', color='blue')
ax.plot(x, theory_bounds, label='Theoretical Bound exp(u(min(h,r_c))/sigma)', color='gray', linestyle='--')
ax.vlines([r_c], ymin=1e-2, ymax=1e5, colors='red', linestyles='dashed', label='Saturation point (h = r_c)')
ax.set_yscale('log')
ax.set_ylim(1e-2, 1e5)
ax.set_xlabel('Potential Height')
ax.set_ylabel('Log Exit Time')
ax.set_title('Mean Exit Times from Maximizing Regime')
ax.legend()


## Sensitivity: Noise, Discretization Step, and Particle Count


In [ ]:
import os

import pandas as pd

from run_convergence_diagnostics import BASE_SIGMA, BASE_LR, BASE_N

# Reviewer demand (d): sigma/dt/n sensitivity, reading run_convergence_diagnostics.py's CSV
# directly (that driver deliberately writes no .pkl -- its natural unit is "one tau series per
# (sweep, delta, param)" across three different parameter axes, which doesn't fit
# reconstruct_pkl_files_from_csv's single-x-axis shape).
conv_csv = 'results/master_convergence_diagnostics.csv'

if os.path.exists(conv_csv):
    conv_df = pd.read_csv(conv_csv)

    def _series_with_baseline(df, sweep_name, param_col, base_value):
        """Rows for `sweep_name`, plus the shared 'baseline' rows relabeled to sit at
        `param_col=base_value` -- the deduplicated shared point (2026-08-07) is stored once,
        under its own 'baseline' sweep label, not inside every axis; see
        run_convergence_diagnostics.py's dedup comment."""
        axis = df[df['sweep'] == sweep_name]
        base = df[df['sweep'] == 'baseline'].copy()
        base[param_col] = base_value
        return pd.concat([axis, base], ignore_index=True)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    for delta in sorted(conv_df['delta'].unique()):
        d_df = conv_df[conv_df['delta'] == delta]

        # sigma sweep: sigma*log(mean(tau)) vs sigma -- the empirical analogue of Theorem 3.1's
        # asymptotic statement lim_{sigma->0} sigma*log(E[tau]) = V_beta(...), which should
        # stabilize (not keep growing) as sigma shrinks if the theory's finite-size assumptions
        # hold at these deltas (both already in the saturated h>r_c regime, see cell 5 above).
        sigma_df = _series_with_baseline(d_df, 'sigma', 'sigma', BASE_SIGMA)
        sigma_grouped = sigma_df.groupby('sigma')['tau'].mean().sort_index()
        axes[0].plot(sigma_grouped.index, sigma_grouped.index * np.log(sigma_grouped.values),
                     marker='o', label=f'delta={delta}')

        # lr sweep: mean(tau) vs discretization step -- tau is already continuous physical time
        # (first_entry_time returns i*lr), so a genuine continuous-time limit should show this
        # roughly flattening as lr shrinks, not diverging.
        lr_df = _series_with_baseline(d_df, 'lr', 'lr', BASE_LR)
        lr_grouped = lr_df.groupby('lr')['tau'].mean().sort_index()
        axes[1].plot(lr_grouped.index, lr_grouped.values, marker='o', label=f'delta={delta}')

        # n sweep: mean(tau) vs particle count -- more particles should only ever help (more
        # independent attempts to be the one that escapes first).
        n_df = _series_with_baseline(d_df, 'n_particles', 'n', BASE_N)
        n_grouped = n_df.groupby('n')['tau'].mean().sort_index()
        axes[2].plot(n_grouped.index, n_grouped.values, marker='o', label=f'delta={delta}')

    axes[0].set_xlabel('sigma'); axes[0].set_ylabel('sigma * log(mean(tau))')
    axes[0].set_title('Sensitivity: noise sigma'); axes[0].legend(); axes[0].grid(True)

    axes[1].set_xlabel('lr (discretization step)'); axes[1].set_ylabel('mean(tau)')
    axes[1].set_yscale('log')
    axes[1].set_title('Sensitivity: discretization step'); axes[1].legend(); axes[1].grid(True)

    axes[2].set_xlabel('n (particle count)'); axes[2].set_ylabel('mean(tau)')
    axes[2].set_title('Sensitivity: particle count'); axes[2].legend(); axes[2].grid(True)

    plt.tight_layout()
    plt.show()
else:
    print(f"{conv_csv} not found -- run `python run_convergence_diagnostics.py` on the cluster "
          "first, then re-run this cell.")


## 1 Dimensional Double Well Potential

### Plots

In [ ]:
from run_experiments_resumable import DOUBLE_WELL_TILT, ALGO_NAMES as WELL_ALGO_NAMES
from viz import baseline_series

x = []
for delta in range(1, 31):
    kwargs = asymmetric_double_well(delta, DOUBLE_WELL_TILT)
    x.append(kwargs['critical_points'][2] - kwargs['critical_points'][0])

# All 8 baselines (2026-08-07): CBO Annealed / Standard CBO / Standard PSO are the
# reviewer-requested fairness and gradient-free comparisons, Langevin is the fixed-noise control
# with no hyperparameter to tune -- see run_experiments_resumable.py. SoftMin Decay stays excluded
# (see Remark rem:higher-dim: a properly-tuned fixed beta consistently beat every decaying/growing
# schedule tested, gap widening at harder barriers).
#
# Masking is now status-based (status_filename_template), not a numeric mask_below cutoff: a run
# that never escapes can't drag a point's mean toward the max_steps*lr timeout sentinel. The old
# mask_below=1e4 silently stopped matching once MAX_STEPS/lr changed (plan's B3 finding) --
# status-based masking can't go stale the same way.
series = baseline_series(
    WELL_ALGO_NAMES, '{a}_double_well_{p}.pkl', '{a}_double_well_{p}_status.pkl',
)

fig, ax = plot_entry_time_comparison(
    x=x, results_dir='results', series=series, params=range(1, 31),
    xlabel='Potential Barrier Height', title='Mean Entry Time vs Potential', grid=True,
)
plt.show()


## 2 Dimensional Quadruple Well Potential

In [ ]:
from run_experiments_resumable import ALGO_NAMES as WELL_ALGO_NAMES
from viz import baseline_series

# depth_gap = f(shallow diagonal well) - f(deep anti-diagonal well), replacing the pre-2026-08-07
# sorted-critical-f-value proxy (which relied on unbalanced_quadruple_well_setting's now-removed
# `critical_points` -- an expensive per-call sympy solve, dropped since nothing downstream read
# it once this figure's own x-axis was redesigned; see potentials.py). depth_gap grows linearly
# with A (exactly -A*delta = A at the fixed delta=-1, verified numerically).
#
# This also fixes B1/B1b: the benchmark's 'particle_init'/'set' were previously an EXACT mirror
# pair (identical depth, nothing suboptimal to escape from -- the same defect
# asymmetric_double_well's tilt term fixes for the 1D benchmark) due to a radius-formula bug.
# Fixed 2026-08-07: 'particle_init' now starts in the genuinely shallower diagonal well, 'set'
# targets the genuinely deeper anti-diagonal well. Pre-2026-08-07 quadruple_well results are not
# comparable to this figure.
x_ = [unbalanced_quadruple_well_setting(A, -1)['depth_gap'] for A in range(2, 31)]

# All 8 baselines (2026-08-07) -- CBO/PSO were previously excluded from this figure entirely,
# which is specifically what reviewer Ssem's Figure 1 (Middle) complaint was about. See the
# double-well cell above for the masking/baseline-set rationale (identical here).
series = baseline_series(
    WELL_ALGO_NAMES, '{a}_quadruple_well_{p}.pkl', '{a}_quadruple_well_{p}_status.pkl',
)

fig, ax = plot_entry_time_comparison(
    x=x_, results_dir='results', series=series, params=range(2, 31),
    xlabel='Depth Gap: f(shallow well) - f(deep well)', title='Mean Entry Time vs Potential', grid=True,
)
plt.show()


## Hgh Dimensional Ackley Function

In [ ]:
from run_ackley_dimensions import ALGO_NAMES as ACKLEY_ALGO_NAMES, FILENAME_BY_ALGO
from viz import baseline_series

x = list(range(10, 100, 10))

# All 8 baselines (2026-08-07), including Langevin -- see the double-well cell's comment for the
# general rationale. FILENAME_BY_ALGO is imported directly (not re-derived) since each algo's
# filename prefix is capitalized inconsistently (SoftMin_, CBO_, StandardCBO_, ...), not a uniform
# pattern -- re-deriving it here would risk silently drifting from what the driver actually
# writes. .removeprefix('results/') because plot_entry_time_comparison already joins results_dir.
_ackley_filenames = {a: t.replace('{dim}', '{p}').removeprefix('results/')
                      for a, t in FILENAME_BY_ALGO.items()}
series = baseline_series(
    ACKLEY_ALGO_NAMES, _ackley_filenames, '{a}_ackley_function_dim{p}_status.pkl',
)

fig, ax = plot_entry_time_comparison(
    x=x, results_dir='results', series=series,
    xlabel='Dimension', title='Mean Entry Time vs Dimension for Ackley Function', grid=True,
)
plt.show()


## Lennard-Jones Cluster Minimization

Real-physics, exact-gradient, NP-hard global-optimization benchmark (d=39..225, beyond the Ackley range above). Global minima are literature constants from the Cambridge Cluster Database (Wales & Doye 1997), `potentials.LJ_GLOBAL_MINIMA`. N=38 is the canonical hard double-funnel case (icosahedral local funnel vs. the true fcc-truncated-octahedron global minimum) -- see the driver's verification notes. Each algorithm's `first_value_time` hit time and the L-BFGS-B-quenched energy of its best configuration are both reported (see `run_lennard_jones.py`).

In [ ]:
from run_lennard_jones import ALGO_NAMES as LJ_ALGO_NAMES
from viz import baseline_series

x = [13, 38, 55, 75]

series = baseline_series(
    LJ_ALGO_NAMES, '{a}_t_lennard_jones_n{p}.pkl', '{a}_status_lennard_jones_n{p}.pkl',
)

fig, ax = plot_entry_time_comparison(
    x=x, results_dir='results', series=series,
    xlabel='Number of Atoms (N)', title='Mean First-Value Hit Time vs Cluster Size (Lennard-Jones)',
    grid=True,
)
plt.show()


In [ ]:
from run_lennard_jones import ALGO_NAMES as LJ_ALGO_NAMES
from viz import baseline_series

x = [13, 38, 55, 75]

series = baseline_series(LJ_ALGO_NAMES, '{a}_e_lennard_jones_n{p}.pkl')

# Subtract the literature global minimum (keyed by N) so the y-axis is a gap above the true
# optimum, comparable across cluster sizes -- not a raw (very negative) energy.
fig, ax = plot_value_comparison(
    x=x, results_dir='results', series=series,
    xlabel='Number of Atoms (N)', ylabel='Quenched Energy Gap above Global Minimum',
    title='Quenched Energy Gap vs Cluster Size (Lennard-Jones)', grid=True,
    value_transform=lambda values, p: values - LJ_GLOBAL_MINIMA[p],
)
plt.show()


## Computational Cost Comparison

Evaluation-count-based cost comparison (gradient/objective oracle calls), per the reviewers' request for cost-matched comparisons rather than raw per-iteration counts. Fully local -- no cluster run needed.

In [ ]:
import os

from viz import BASELINE_STYLE
from run_experiments_resumable import ALGO_NAMES as WELL_ALGO_NAMES

# Real cost data (2026-08-07) from the actual double_well/quadruple_well sweep, not a separate
# 2000-step, d=2, single-seed probe -- see the plan's reviewer demand (b) finding: the old probe
# used _gevals/_oevals counters that (pre-2026-08-07) counted one per vectorized CALL, not one per
# particle, undercounting by a factor of n=100. Simulation._eval_grad/_eval_obj now count
# per-particle, and every driver logs {algo}_gevals/{algo}_oevals/{algo}_secs directly.
cost_csv = 'results/master_experiment_log_decay.csv'

if os.path.exists(cost_csv):
    cost_df_raw = pd.read_csv(cost_csv)
    rows = []
    for a in WELL_ALGO_NAMES:
        label = BASELINE_STYLE[a][0]
        rows.append((
            label,
            cost_df_raw[f'{a}_gevals'].mean(),
            cost_df_raw[f'{a}_oevals'].mean(),
            cost_df_raw[f'{a}_secs'].mean(),
        ))
    cost_df = pd.DataFrame(rows, columns=[
        'Method', 'Mean Gradient Evaluations (particle-steps)',
        'Mean Objective Evaluations (particle-steps)', 'Mean Wall-clock Time (s)',
    ])
    cost_df
else:
    print(f"{cost_csv} not found -- run `python run_experiments_resumable.py` on the cluster "
          "first, then re-run this cell. Counts are per-particle (n_particles x steps, not "
          "per-vectorized-call) and averaged across the full double_well/quadruple_well sweep "
          "actually run, unlike the old 2000-step d=2 single-seed probe this cell used to use.")


## Ascent-Descent Dynamics

Illustrates the sign-changing effective learning rate $A_{t,k}$ (Section 2): particles below the Softmin threshold descend ($A_{t,k}>0$), particles above it ascend/explore ($A_{t,k}<0$). Fully local -- a short single-seed run, no cluster dependency.

In [ ]:
from run_experiments_resumable import DOUBLE_WELL_TILT

np.random.seed(0)
setting = asymmetric_double_well(4, DOUBLE_WELL_TILT)
demo_sim = SoftminEnergyAdagrad(
    n=20, d=1, f=setting['f_vec'], df=setting['f_grad_vec'],
    particle_init=setting['particle_init'], bounds=setting['bounds'],
    initial_lr=0.01, adaptive_lr=False, noise_intensity=1.0, beta=10.0,
    noise="normal", track_prefactor_history=True,
)
demo_sim.run(500)
fig, ax = plot_prefactor_dynamics(demo_sim)
plt.show()


## Ablation: Ascent-Descent Mechanism

Compares the real (signed) mechanism against two ablations from the revised theory's placeholder: replacing $A_{t,k}$ by $(A_{t,k})_+$ (ascending particles freeze) and by $|A_{t,k}|$ (every particle always descends). Kept separate from the SoftMin-vs-SA-vs-CBO-vs-PSO comparisons above. Needs `python run_ablation_experiment.py` run on the cluster first; the cell below prints a message instead of failing if that hasn't happened yet.

In [ ]:
import os

ablation_dir = 'results'
sample_file = os.path.join(ablation_dir, 'ablation_signed_double_well_1.pkl')

if os.path.exists(sample_file):
    _ablation_style = {
        'signed': ('Signed (A_t,k)', '#0072B2', 'o'),
        'positive_part': ('Positive part (A_t,k)_+', '#E69F00', 's'),
        'absolute': ('Absolute value |A_t,k|', '#009E73', '^'),
    }
    ablation_series = [
        {'label': label, 'filename_template': f'ablation_{mode}_double_well_{{p}}.pkl',
         'status_filename_template': f'ablation_{mode}_double_well_{{p}}_status.pkl',
         'color': color, 'marker': marker}
        for mode, (label, color, marker) in _ablation_style.items()
    ]
    fig, ax = plot_entry_time_comparison(
        x=range(1, 31), results_dir=ablation_dir, series=ablation_series,
        xlabel='Potential Barrier Height (delta)', title='Ascent-Descent Mechanism Ablation', grid=True,
    )
    plt.show()
else:
    print("Ablation results not found in 'results/' -- run `python run_ablation_experiment.py` "
          "on the cluster first, then re-run this cell.")


## Hyperparameter Sensitivity: Softmin Beta


In [ ]:
import os

import pandas as pd

# Reviewer demand (d): the beta-sensitivity study behind Softmin's shipped beta=10 (see
# hyperparameter_diagnostics.py) is now persisted to CSV via write_escape_results_csv, instead of
# stdout-only -- citable directly from this cell rather than needing to re-run that script and
# read terminal output.
beta_csv = 'results/hyperparameter_diagnostics_beta_sweep.csv'

if os.path.exists(beta_csv):
    beta_df = pd.read_csv(beta_csv)
    fig, ax = plt.subplots(figsize=(10, 6))
    for benchmark in beta_df['benchmark'].unique():
        sub = beta_df[beta_df['benchmark'] == benchmark]
        ax.scatter(sub['candidate'], sub['entry_time'], label=benchmark, s=60)
    ax.set_xlabel('beta candidate (fixed value or annealing schedule)')
    ax.set_ylabel('entry_time')
    ax.set_yscale('log')
    ax.set_title('Softmin Beta Sensitivity (double_well / quadruple_well / Ackley midpoints)')
    ax.legend()
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print(f"{beta_csv} not found -- run `python hyperparameter_diagnostics.py` first, then "
          "re-run this cell.")
